# Molab 02: Confirmatory Sweep, Leakage Check & Alert Policy (NB4)

**Goal:** Run the boosted-tree confirmatory sweep that tests the frozen hypotheses C1-C5 on data that did not inform them, and that is the precondition for `LI-Small` (and every other sealed dataset) to be unsealed at all. **This notebook has never been run anywhere in this project before this** — the audit found zero NB4 evidence in any of the four PCs' work.

**Covers:** NB4 in full — the shuffled-label leakage check (runs *first*, and the notebook stops and warns loudly if it fails, exactly as the project's own rule requires), the 9-horizon x 2-protocol x 3-seed LightGBM sweep on HI-Small and LI-Small (tier P0), then SAML-D / HI-Medium / LI-Medium (tier P1) with whatever time remains. Tier P2 (rolling-origin folds plus the Ethereum dataset) is not run by default: P2 also includes the Ethereum sweep, which needs `MulDiGraph.pkl` — not downloadable by `amlh.fetch.ensure_raw` (see `fetch.py`), so it would break this notebook's self-contained design. Add `'P2'` to `TIERS` and attach that file by hand if you want those jobs too.

**Why HI-Small's own numbers here are not a new confirmatory result:** `docs/DECISIONS.md` D-001 is explicit — "HI-Small is the discovery set. A claim counts as confirmed only on data not modelled when the hypotheses were frozen." This notebook's HI-Small sweep is a faithful reproduction check (useful for catching regressions, and it is shown below), not evidence for C1-C5. **LI-Small is the real target**: it is sealed (`amlh.sealing.SEALED`), so this notebook deliberately never computes or displays its test metrics — only validation metrics and label-free health checks, exactly as `gbt_sweep`'s own code already enforces. Completing LI-Small's full sweep here is what lets notebook 03's merge step unseal it and compute this project's **first real confirmatory verdicts**.

**Expected wall time on the stated hardware:** CPU-only: LightGBM at the frozen, un-tuned hyperparameters (`amlh.jobs_gbt.LGB_PARAMS`, fixed by DECISIONS.md D-001) on 4 cores. 2 to 5 hours for the full HI-Small + LI-Small P0 sweep (114 jobs total; the project's own measurement on a similar-class CPU was 1 to 2 minutes per HI-Small job including features), leaving most of a 12 h session for the P1 datasets. **No GPU needed or used** — the frozen protocol is CPU LightGBM by design (DECISIONS.md D-001: switching LightGBM to a GPU build would change a frozen detector). Runs correctly on Molab with the GPU detached.

## What makes this notebook self-contained
Every cell below is idempotent and safe to re-run. On a fresh Molab session it, in order: (1) clones `aml-stream`'s
`main` branch from GitHub (public, no token needed — set `GITHUB_TOKEN` as an environment variable/secret only if the
repository has since been made private); (2) downloads the AMLworld CSVs it needs directly from Kaggle's public
download endpoint (no `kaggle.json` needed, with a hash check against `docs/SPEC.md`'s verified values, because one
download endpoint has silently truncated a file before); (3) installs the exact pinned dependencies; (4) runs the
test suite as a gate — **a failing test stops the notebook before it spends any compute**; (5) reuses whatever
earlier, *verified* output already exists in the cloned repository (so PC-8's and PC-18's already-finished jobs are
never repeated); (6) runs its job list under a wall-clock budget that always stops cleanly before the 12 h session
limit, in a way that is safe to interrupt and resume (`Run all` again after a restart continues from the last
checkpoint, it does not start over); (7) writes a human-readable analysis with tables and plots; (8) zips the output
folder so it survives the session ending.

**Nothing here needs you to upload or attach anything by hand.**

In [ ]:
# ============================= Settings: the only cell you would normally change =============================
import os, sys, time
NOTEBOOK = 'NB4'
REPO_URL = os.environ.get('AML_REPO_URL', 'https://github.com/Shahoud867/aml-stream.git')
REPO_BRANCH = os.environ.get('AML_REPO_BRANCH', 'main')   # pin a commit instead by setting this to a SHA
BUDGET_HOURS = float(os.environ.get('AML_BUDGET_HOURS', 11.5))   # stops cleanly well inside Molab's 12 h cap
TIERS = ('P0', 'P1')
RUN_TESTS = True     # a failing test stops the notebook before it spends any compute
INSTALL = True

SESSION_START = time.time()
def hours_left():
    return BUDGET_HOURS - (time.time() - SESSION_START) / 3600


In [ ]:
# ============================= Where things live =============================
# A plain, dependency-free layout that works on any Jupyter kernel (Molab, a local machine, Colab, ...): everything
# lives under the user's home directory, so no platform-specific mount point is assumed.
from pathlib import Path

BASE = Path(os.environ.get('AML_WORKDIR', Path.home() / 'aml_molab'))
BASE.mkdir(parents=True, exist_ok=True)
CODE = BASE / 'aml-stream'               # the cloned repository
INPUT = BASE / 'data'                    # raw AMLworld files
CACHE = BASE / 'cache'                   # prepared (parquet) data, rebuilt once and reused across cells
WORKING = BASE / f'out_{NOTEBOOK.lower()}'
WORKING.mkdir(parents=True, exist_ok=True)
for p in (INPUT, CACHE):
    p.mkdir(parents=True, exist_ok=True)
print('workdir', BASE)
print('  code   ->', CODE)
print('  data   ->', INPUT)
print('  cache  ->', CACHE)
print('  output ->', WORKING)

In [ ]:
# ============================= Get the code =============================
# Clones (or fast-forwards) the repository. The expected code hash is computed from whatever we actually cloned,
# never hard-coded, so this notebook can never go stale the way a fixed-hash code package would.
import subprocess

def run(cmd, **kw):
    r = subprocess.run(cmd, text=True, capture_output=True, **kw)
    if r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stdout}\n{r.stderr}")
    return r.stdout

token = os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN')
clone_url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
if not (CODE / '.git').exists():
    print('cloning', REPO_URL, '...')
    run(['git', 'clone', '--branch', REPO_BRANCH, clone_url, str(CODE)])
else:
    print('updating existing clone ...')
    run(['git', '-C', str(CODE), 'fetch', 'origin', REPO_BRANCH])
    run(['git', '-C', str(CODE), 'checkout', REPO_BRANCH])
    run(['git', '-C', str(CODE), 'reset', '--hard', f'origin/{REPO_BRANCH}'])

sys.path.insert(0, str(CODE / 'src'))
from amlh.runner import code_hash
CODE_HASH = code_hash(CODE)
print('HEAD', run(['git', '-C', str(CODE), 'rev-parse', 'HEAD']).strip())
print('code hash', CODE_HASH)

In [ ]:
# ============================= Install dependencies =============================
if INSTALL:
    run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements-kaggle.txt')])
    print('requirements-kaggle.txt installed')


In [ ]:
# ============================= Raw data =============================
# Downloaded directly from Kaggle's public endpoint (no account needed) and checked against the row counts and
# SHA-256 hashes in docs/SPEC.md -- the same check every notebook in this project runs, because one Kaggle download
# endpoint has silently truncated a file before while still reporting success.
from amlh.fetch import ensure_raw
from amlh.session import verify_raw

DATASETS = ['HI-Small', 'LI-Small', 'SAML-D', 'HI-Medium', 'LI-Medium']  # every P1 dataset; TIERS below stops at P1, not P2, because Ethereum (P2) needs a file this notebook cannot fetch on its own
print('fetching', DATASETS)
ensure_raw(DATASETS, INPUT)
for d in DATASETS:
    print(d, verify_raw(d, INPUT))

In [ ]:
# ============================= Tests: the gate =============================
from amlh.session import run_preflight_tests
if RUN_TESTS:
    run_preflight_tests(CODE)
    print('all tests passed')

In [ ]:
# ============================= Reuse prior verified work =============================
# The freshly cloned repository's own results/ folder holds every job this project has finished so far, in exactly
# the job-folder layout the runner understands. Pointing it there for free: a job already verified elsewhere is
# never repeated, and a job that was interrupted partway (with a usable checkpoint) resumes instead of restarting.
PRIOR_RESULTS = [CODE / 'results']
print('will reuse any verified job found under', [str(p) for p in PRIOR_RESULTS])

In [ ]:
# ============================= Run long jobs in short, yielding slices =============================
# amlh.session.run_notebook's own job loop is a plain synchronous `for` loop: gbt_sweep (LightGBM) and the
# polling loop around an isolated mgnn_train/tdgnn_train subprocess both run entirely in-process, with no
# `await` anywhere inside them. A single call that runs for a long time therefore blocks this kernel's own
# async event loop for that whole time. Observed live, pairing this exact notebook with the marimo-pair skill:
# a long call like this gets cut short after an inconsistent, short burst (anywhere from under a minute to
# several minutes), confirmed from the runner's own ledger to be `stop_requested` becoming true (not this
# project's own time-budget check: remaining() was still far above what the next step needed) -- consistent
# with something outside this project's code deciding the kernel looked unresponsive and signalling it.
#
# The fix is structural, not a workaround bolted onto the symptom: never make one long blocking call. This
# calls run_notebook repeatedly in short slices, with a real `await asyncio.sleep(...)` between each one, so
# the event loop actually gets a turn regularly. run_notebook is explicitly designed to be called again and
# again against the same working directory -- a finished job is never repeated, and a job that only got partway
# resumes from its checkpoint -- so slicing it changes nothing about correctness, only how the blocking is
# shaped. This is harmless, and barely slower in the ordinary case, even if this diagnosis turns out wrong.
import asyncio
import time as _time
from amlh.session import run_notebook

SLICE_HOURS = 0.05   # about 3 minutes of real work per slice before yielding


async def run_notebook_sliced(notebook, budget_hours, restore_roots=None, **kw):
    # run_notebook() does two things fresh on every call that are each real work, not a cheap check, when
    # repeated every few minutes instead of once: (1) restore_roots -- Runner.restore() verifies (sha256-hashes)
    # every file of every already-finished job under every root; this project's whole results/ tree (NB3 alone
    # has 90 jobs with checkpoints and score files) is a lot to re-hash. (2) check_raw -- verify_raw reads the
    # *entire* raw CSV to count its rows even with verify_hashes=False, and HI-Medium/LI-Medium are each several
    # GB. Passing both fresh every slice meant each ~3-minute slice spent its whole budget redoing both of these
    # from scratch, leaving no time to ever run a new job -- confirmed live: DONE froze for dozens of consecutive
    # slices while the budget kept draining. Both only need to happen once, on the first slice: later slices'
    # Runner already has the restored files in its own working directory, and the raw file has not changed.
    t0 = _time.time()
    summary = {'complete': False}
    first = True
    while not summary.get('complete'):
        left = budget_hours - (_time.time() - t0) / 3600
        if left <= 0:
            break
        summary = run_notebook(notebook, budget_hours=min(SLICE_HOURS, left), verify_hashes=False, check_raw=first,
                                restore_roots=(restore_roots if first else []), **kw)
        first = False
        print(f"  slice done: {summary.get('counts', {})}, {left:.2f} h left in this call's own budget")
        await asyncio.sleep(1)   # the actual yield: without this, the next slice starts before the loop ever turns
    return summary

## Stage 0: the leakage check — runs before anything else

A detector trained on **shuffled labels** must score at prevalence (it has learned nothing real). If it scores well
above prevalence instead, the feature pipeline is leaking the label through some other column, and **every
positive-horizon result in this whole project would be invalid** until that is fixed. This is deliberately the very
first thing this notebook computes, with no other work started first.

In [ ]:
from amlh.runner import Runner
from amlh.plans import plan, registry
from amlh.jobs_gbt import Paths

Paths.raw = {'HI-Small': INPUT, 'LI-Small': INPUT}
Paths.cache = CACHE

leakage_jobs = [j for j in plan('NB4') if 'shuffled' in j.job_id]
print(f'{len(leakage_jobs)} leakage-check jobs:', [j.job_id for j in leakage_jobs])

leak_run = WORKING / 'leakage_check'
r = Runner(leak_run, 'NB4-leakage', 3600, code_version=CODE_HASH, registry=registry())
r.install_signal_handlers()
r.restore(PRIOR_RESULTS)
r.write_manifest({'stage': 'leakage_check'})
summary = r.run(leakage_jobs, tiers=('P0',))
print(summary)


In [ ]:
import numpy as np
from amlh.merge import collect, leakage_check
from amlh.jobs_gbt import get_stream

c = collect([leak_run] + PRIOR_RESULTS)
failed = []
for ds in ('HI-Small',):   # the frozen leakage-check jobs are defined on HI-Small (plans.py nb4())
    s, _ = get_stream({'dataset': ds, 'protocol': 'released'})
    res = leakage_check(c, s)
    for jid, v in res.items():
        status = 'PASS' if v['pass'] else 'FAIL'
        print(f"{jid}: core PR-AUC {v['core_pr_auc']:.5f} vs {10 * v['prevalence']:.5f} (10x prevalence) -> {status}")
        if not v['pass']:
            failed.append(jid)

assert not failed, (
    f"LEAKAGE CHECK FAILED for {failed}. Per this project's own rule, do not run anything below this cell. "
    "Investigate the feature pipeline (src/amlh/design.py, src/amlh/features.py) before trusting any positive-"
    "horizon result anywhere in this project."
)
print()
print('Leakage check passed. Continuing to the main sweep.')


## The main sweep, alert policy, and (time permitting) the Medium/SAML-D/fold tiers

Reuses `amlh.session.run_notebook` exactly as the Kaggle/Colab NB4 does: per-job checkpointing (a LightGBM fit that
is interrupted restarts that one fit, not the whole sweep), a wall-clock budget that stops cleanly, and automatic
tier ordering (P0 before P1 before P2) so the most important deliverable — the full HI-Small + LI-Small P0
sweep and alert policy — is always attempted first.

In [ ]:
from amlh.session import print_summary, final_checklist

nb4_work = WORKING / 'nb4'
summary = await run_notebook_sliced('NB4', budget_hours=hours_left(), tiers=TIERS, input_root=INPUT, working=nb4_work,
                                     restore_roots=[leak_run] + PRIOR_RESULTS, cache_root=CACHE, run_name='aml_run')
print_summary(nb4_work)
final_checklist('NB4', TIERS, nb4_work, platform='molab')


## Results: hypothesis verdicts

**HI-Small** (exploratory / reproduction check only, per D-001 above). **LI-Small**: validation metrics and
label-free health only — if you see no test numbers for LI-Small below, that is the sealing rule working as
designed, not a bug.

In [ ]:
import json
from amlh.merge import collect, sweep_results, verdicts_for
from amlh.jobs_gbt import get_stream

c = collect([nb4_work, leak_run] + PRIOR_RESULTS)
print(f"{len(c.jobs)} verified jobs total; {len(c.problems)} problems")

s, _ = get_stream({'dataset': 'HI-Small', 'protocol': 'released'})
rows = sweep_results(c, s)
if rows:
    v = verdicts_for(c, s, rows)
    print('HI-Small (exploratory reproduction, NOT a confirmatory result -- see markdown above):')
    for hid, val in sorted(v.items()):
        value = val.get('value')
        print(f"  {hid}: {value:.4f} " + ('PASS' if val.get('pass') else 'FAIL') if value is not None else f"  {hid}: {val}")
    (WORKING / 'hi_small_verdicts.json').write_text(json.dumps(v, indent=1, default=str))

    import matplotlib.pyplot as plt
    import numpy as np
    HORDER = ['0', '1', '60', '600', '3600', '21600', '86400', '259200', 'inf']
    seeds = sorted({k for _, k in rows})
    curve = [(h, np.mean([rows[(h, k)]['row']['test_core']['pr_auc'] for k in seeds if (h, k) in rows]))
             for h in HORDER if any((h, k) in rows for k in seeds)]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(range(len(curve)), [c[1] for c in curve], 'o-', color='tab:orange')
    ax.set_xticks(range(len(curve))); ax.set_xticklabels([c[0] for c in curve], rotation=45)
    ax.set_xlabel('decision horizon H'); ax.set_ylabel('core PR-AUC')
    ax.set_title('LightGBM, HI-Small: core PR-AUC vs horizon (reproduction check)')
    plt.tight_layout(); plt.savefig(WORKING / 'hi_small_horizon_curve.png', dpi=130); plt.show()
else:
    print('HI-Small sweep not yet complete in this session.')

li, _ = get_stream({'dataset': 'LI-Small', 'protocol': 'released'})
li_jobs = [jid for jid in c.jobs if '-gbt-wall-LI-Small-released-H' in jid and 'fold' not in jid and 'shuffled' not in jid]
print()
print(f"LI-Small: {len(li_jobs)} of 54 planned P0 sweep jobs verified (needed for notebook 03 to unseal it).")
print('No test metric for LI-Small is computed here -- that only ever happens once, in the merge step.')


In [ ]:
# ============================= Package the output =============================
# Molab sessions are not guaranteed to persist, so the output is zipped here (and the notebook's own cell outputs,
# saved with the .ipynb itself, carry the printed tables below even if the zip is never downloaded).
import shutil
archive = shutil.make_archive(str(BASE / f'confirmatory_sweep_{NOTEBOOK.lower()}'), 'zip', WORKING)
print('wrote', archive, f'({Path(archive).stat().st_size / 1e6:.1f} MB)')
print()
print('Download this file before your Molab session ends. If you plan to run another of the three notebooks next,')
print('the most useful thing to carry over is the whole', WORKING, 'folder (or this zip, unpacked next to the new')
print("notebook's own working directory) -- the next notebook's merge step will pick up anything placed under")
print("its PRIOR_RESULTS / EXTRA_ROOTS list.")

## What to do next

- If `hours_left()` is still positive and jobs remain (check the summary above), `Run all` again: nothing finished
  is repeated.
- Once LI-Small's P0 sweep (54 jobs) and its alert policy (3 jobs) are complete, run **Notebook 03** — its
  merge step will unseal LI-Small for the first time in this project and report the real C1-C5 verdicts. Point its
  `EXTRA_ROOTS` at this notebook's output (the zip above, unzipped) so it can see this work.